In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


#from pyspark.sql.functions import (
#    input_file_name,
#    current_timestamp,
#    regexp_extract
#)

In [0]:
 %python
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "etl_vinkos")
dbutils.widgets.text("esquema", "silver")
dbutils.widgets.text("storageName", "adlsvinkosetl")
dbutils.widgets.text("esquemagold", "gold")


In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
esquemagold = dbutils.widgets.get("esquemagold")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/*.txt"

print(ruta)

abfss://raw@adlsvinkosetl.dfs.core.windows.net/*.txt


In [0]:

#Lee los archivos txt y agrupamos los nombres y totales bitacora_ctrl_file
from pyspark.sql.functions import *


from pyspark.sql.functions import *


# Validar si existen archivos TXT en RAW

#ruta_raw = f"abfss://{container}@{storageName}.dfs.core.windows.net/"

try:

    archivos = dbutils.fs.ls(ruta)

    archivos_txt = [
        a for a in archivos
        if a.name.endswith(".txt")
    ]

    if len(archivos_txt) == 0:
        print("No existen archivos para procesar.")
        dbutils.notebook.exit("SIN_ARCHIVOS")

except Exception:

    print("No existen archivos para procesar.")
    dbutils.notebook.exit("SIN_ARCHIVOS")

# Lectura de todos los TXT


df = (
    spark.read
         .option("header", "true")
         .csv(ruta)
         .withColumn(
             "archivo_origen",
             regexp_extract(
                 input_file_name(),
                 r'([^/]+$)',
                 1
             )
         )
         .withColumn(
             "fecha_carga",
             current_timestamp()
         )
)

display(df)

email,fgh,Badmail,Baja,Fecha envio,Fecha open,Opens,Opens virales,Fecha click,Clicks,Clicks virales,Links,IPs,Navegadores,Plataformas,archivo_origen,fecha_carga
angelica-daniel@live.com.mx,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
sexi_barbi991@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
abrilgocar@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
pm_andres10@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
dolphin2006_1@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
chasada_10@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
clonroterdam@yahoo.com.mx,null,HARD,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
adri_gonz671@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
chaparritagallinita3@yahoo.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z
pruebita.software@hotmail.com,null,null,null,14/02/2013 17:35,-,0,0,-,0,0,-,-,-,-,report_9.txt,2026-07-08T06:11:03.582945Z


In [0]:
from pyspark.sql.functions import count
#Crea el df con los nombres de los archivos que no estan en la bitacora_ctrl_file
df_bitacora = (
    df.groupBy("archivo_origen")
      .agg(
          count("*").alias("total_registros")
      )
      .withColumnRenamed(
          "archivo_origen",
          "nom_archivo"
      )
      .withColumn(
          "fecha_proceso",
          current_timestamp()
      )
      .withColumn(
          "estatus",
          lit("PROCESADO")
      )
      .withColumn(
          "mensaje_error",
          lit(None).cast("string")
      )
)


In [0]:
#muestra lo que tiene la tabla de bitacora_ctrl_file

df_control = spark.table(
    f"{catalogo}.{esquemagold}.bitacora_ctrl_file"
)

display(df_control)

id_bitacora,nom_archivo,fecha_proceso,total_registros,estatus,mensaje_error


In [0]:
#Hace un anti join y se trae los que no coicidan df_bitacora_nueva

df_bitacora_nueva = (
    df_bitacora.join(
        df_control.select("nom_archivo"),
        on="nom_archivo",
        how="left_anti"
    )
)

display(df_bitacora_nueva)

nom_archivo,total_registros,fecha_proceso,estatus,mensaje_error
report_9.txt,995,2026-07-08T06:11:04.572194Z,PROCESADO,null
report_7.txt,503,2026-07-08T06:11:04.572194Z,PROCESADO,null
report_8.txt,503,2026-07-08T06:11:04.572194Z,PROCESADO,null


In [0]:

# se hace la conversion de total_registros a int
from pyspark.sql.functions import col

df_bitacora_nueva = (
    df_bitacora_nueva
        .withColumn(
            "total_registros",
            col("total_registros").cast("int")
        )
)
display(df_bitacora_nueva)

nom_archivo,total_registros,fecha_proceso,estatus,mensaje_error
report_9.txt,995,2026-07-08T06:11:05.376726Z,PROCESADO,null
report_7.txt,503,2026-07-08T06:11:05.376726Z,PROCESADO,null
report_8.txt,503,2026-07-08T06:11:05.376726Z,PROCESADO,null


In [0]:
#Se escribe en la bitacora los nuevos valores
df_bitacora_nueva.write \
    .mode("overwrite") \
.saveAsTable(f"{catalogo}.{esquema}.archivos_a_procesar")

In [0]:
#Se muestra lo que tiene la bitacora de control

df = spark.table(f"{catalogo}.{esquema}.archivos_a_procesar")

display(df)

nom_archivo,fecha_proceso,total_registros,estatus,mensaje_error
report_9.txt,2026-07-08T06:11:06.461651Z,995,PROCESADO,null
report_7.txt,2026-07-08T06:11:06.461651Z,503,PROCESADO,null
report_8.txt,2026-07-08T06:11:06.461651Z,503,PROCESADO,null
